# Clase 3 · 00 · Preparación del TP de streaming
**Caso:** llegan compras de la plataforma; necesitamos contarlas por ventanas y observar datos tardíos.
Continuamos con el esquema del alumno y las dimensiones de la clase 2.
Esta preparación crea recursos faltantes y no vacía tablas ni checkpoints.

In [ ]:
dbutils.widgets.text("student_id", "alumno01", "01 - Identificador")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "02 - Escala de clases 1 y 2")
dbutils.widgets.dropdown("expected_batch_id", "stream_001",
                         ["stream_001", "stream_002", "stream_003", "stream_004", "stream_005"], "03 - Etapa a validar")
dbutils.widgets.text("job_run_id", "interactive", "04 - ID de ejecución")

In [ ]:
%run ../common/config

In [ ]:
%run ../common/generate_stream_data

In [ ]:
%run ../common/streaming_support

In [ ]:
from pyspark.sql import functions as F
config = CourseConfig(spark.sql("SELECT current_catalog()").first()[0],
                      dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
repetitions = STREAM_REPETITIONS[config.scale]
expected_batch_id = normalize_stream_batch(dbutils.widgets.get("expected_batch_id"))
job_run_id = dbutils.widgets.get("job_run_id").strip()[:100] or "interactive"
spark.conf.set("spark.sql.session.timeZone", "UTC")
create_course_namespace(spark, config)

## 1. Prerrequisitos y dimensiones congeladas
`silver_customers` y `silver_products` deben existir y coincidir con la escala seleccionada.
Copiamos ambas una sola vez a snapshots para que el experimento mantenga el mismo enriquecimiento
incluso al reejecutarse. Después del setup no se deben reemplazar esos snapshots.

In [ ]:
initialize_stream_lab(spark, dbutils, config, repetitions)
require_stream_lab(spark, config)
display(spark.table("stream_lab_settings"))
print("Entrada:", stream_input_directory(config))
print("Checkpoints:", {stage: stream_checkpoint(config, stage)
                       for stage in ["bronze", "quality", "dedup", "gold"]})

## 2. Reconocer los recursos
Los cuatro checkpoints pertenecen a consultas distintas; se conservan entre ejecuciones.
Las tablas fuente Delta reciben sólo append durante el TP. No ejecutar OVERWRITE, DELETE o MERGE sobre ellas.
`stream_query_progress` guarda observabilidad; `stream_run_audit`, validaciones terminadas correctamente.

In [ ]:
%sql
SELECT 'customers_snapshot' AS resource, COUNT(*) AS rows FROM stream_customers_snapshot
UNION ALL
SELECT 'products_snapshot', COUNT(*) FROM stream_products_snapshot

In [ ]:
print("Archivos publicados:", published_stream_batches(dbutils, config))
print("Preparación lista. Publicá sólo stream_001 y creá el Job de cinco tareas.")

## 3. Antes de comenzar
- Usá la misma escala y el mismo `student_id` de las clases anteriores.
- En Free Edition, seleccioná serverless y empezá con la escala que ya utilizaste; no cambies el tamaño de dimensiones.
- Todas las consultas usan `AvailableNow`: procesan lo disponible desde el checkpoint y terminan.
- La primera ingestión comprueba el soporte de Auto Loader del workspace al ejecutar el stream.
- Leé la secuencia de cinco etapas en README. **No publiques todos los archivos juntos.**